In [16]:
import xml.etree.ElementTree as ET
import os
import pandas as pd

def parse_ddi_xml(folder_path):
    data = []
    
    for root_dir, dirs, files in os.walk(folder_path):
        for filename in files:
            if not filename.endswith(".xml"):
                continue
                
            filepath = os.path.join(root_dir, filename)
            tree = ET.parse(filepath)
            root = tree.getroot()
            
            for sentence in root.findall(".//sentence"):
                sentence_text = sentence.get("text", "")
                
                # Get all drug entities in this sentence
                entities = {}
                for entity in sentence.findall("entity"):
                    entities[entity.get("id")] = {
                        "text": entity.get("text"),
                        "type": entity.get("type")
                    }
                
                # Get all pairs
                for pair in sentence.findall("pair"):
                    e1_id = pair.get("e1")
                    e2_id = pair.get("e2")
                    ddi = pair.get("ddi")
                    ddi_type = pair.get("type", "NO-REL")
                    
                    if e1_id not in entities or e2_id not in entities:
                        continue
                    
                    e1_text = entities[e1_id]["text"]
                    e2_text = entities[e2_id]["text"]
                    
                    # If ddi is false, it's a negative example
                    if ddi == "false":
                        relation = "NO-REL"
                    else:
                        relation = ddi_type if ddi_type else "NO-REL"
                    
                    data.append({
                        "sentence": sentence_text,
                        "entity1": e1_text,
                        "entity2": e2_text,
                        "relation": relation
                    })
    
    return pd.DataFrame(data)

# Parse train and test
train_df = parse_ddi_xml("../data/ddi_corpus/Train")
test_df = parse_ddi_xml("../data/ddi_corpus/Test")

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))
print("\nSample row:")
print(train_df.head(3))

Train samples: 27792
Test samples: 6657

Sample row:
                                            sentence          entity1  \
0  Differential regulation of tyrosine phosphoryl...  contortrostatin   
1  Differential regulation of tyrosine phosphoryl...  contortrostatin   
2  Differential regulation of tyrosine phosphoryl...       echistatin   

      entity2 relation  
0  echistatin   NO-REL  
1  flavoridin   NO-REL  
2  flavoridin   NO-REL  


In [17]:
from collections import Counter

print("=== Train Relation Distribution ===\n")
train_counts = Counter(train_df["relation"])
total = len(train_df)
for rel, count in sorted(train_counts.items(), key=lambda x: -x[1]):
    print(f"{rel:20} → {count:5} ({count/total*100:.1f}%)")
print(f"\nTotal pairs: {total}")

print("\n=== Test Relation Distribution ===\n")
test_counts = Counter(test_df["relation"])
total_test = len(test_df)
for rel, count in sorted(test_counts.items(), key=lambda x: -x[1]):
    print(f"{rel:20} → {count:5} ({count/total_test*100:.1f}%)")
print(f"\nTotal pairs: {total_test}")

=== Train Relation Distribution ===

NO-REL               → 23772 (85.5%)
effect               →  1687 (6.1%)
mechanism            →  1319 (4.7%)
advise               →   826 (3.0%)
int                  →   188 (0.7%)

Total pairs: 27792

=== Test Relation Distribution ===

NO-REL               →  5678 (85.3%)
effect               →   360 (5.4%)
mechanism            →   302 (4.5%)
advise               →   221 (3.3%)
int                  →    96 (1.4%)

Total pairs: 6657


mechanism — one drug changes how another drug works biochemically. "Aspirin inhibits the metabolism of Warfarin."
effect — one drug changes the clinical effect of another. "Alcohol increases the sedative effect of Diazepam."
advise — a recommendation about using two drugs together. "Avoid combining MAOIs with SSRIs."
int — a general interaction is stated without specifics. "Drug A and Drug B interact."

In [19]:
print("=== Sample Examples Per Relation Type ===\n")

for rel_type in train_df["relation"].unique():
    samples = train_df[train_df["relation"] == rel_type].head(2)
    print(f"--- {rel_type} ---")
    for _, row in samples.iterrows():
        print(f"  E1: {row['entity1']}")
        print(f"  E2: {row['entity2']}")
        print(f"  Sentence: {row['sentence'][:120]}")
        print()

=== Sample Examples Per Relation Type ===

--- NO-REL ---
  E1: contortrostatin
  E2: echistatin
  Sentence: Differential regulation of tyrosine phosphorylation in tumor cells by contortrostatin, a homodimeric disintegrin, and mo

  E1: contortrostatin
  E2: flavoridin
  Sentence: Differential regulation of tyrosine phosphorylation in tumor cells by contortrostatin, a homodimeric disintegrin, and mo

--- effect ---
  E1: Echistatin
  E2: contortrostatin
  Sentence: Echistatin alone had no effect on tyrosine phosphorylation in T24 cells, but dose-dependently inhibits the effects of co

  E1: Flavoridin
  E2: contortrostatin
  Sentence: Flavoridin alone was found to have no effect on CAS, but can completely block contortrostatin-induced phosphorylation of

--- mechanism ---
  E1: cobalt
  E2: iron
  Sentence: Interactions of cobalt and iron in absorption and retention.


  E1: iron
  E2: cobalt
  Sentence: Additional iron significantly inhibited the absorption of cobalt in both dietary c

In [20]:
train_df.to_csv("../data/ddi_train.csv", index=False)
test_df.to_csv("../data/ddi_test.csv", index=False)
print("Saved!")
print("Train:", len(train_df), "rows")
print("Test:", len(test_df), "rows")

Saved!
Train: 27792 rows
Test: 6657 rows


In [21]:
label2id = {
    "NO-REL": 0,
    "effect": 1,
    "mechanism": 2,
    "advise": 3,
    "int": 4
}

id2label = {v: k for k, v in label2id.items()}

print("Label mapping:")
for label, idx in label2id.items():
    count = len(train_df[train_df["relation"] == label])
    print(f"  {idx} → {label:12} ({count} samples)")

Label mapping:
  0 → NO-REL       (23772 samples)
  1 → effect       (1687 samples)
  2 → mechanism    (1319 samples)
  3 → advise       (826 samples)
  4 → int          (188 samples)
